# Tome C1, Chapter 7: Planarity and Colouring (Python companion)

Exercises first, solutions at the end. We count faces of planar embeddings and check Euler's formula, extract Kuratowski subgraphs from non-planar graphs, run Heawood's five colour algorithm with real Kempe chains on random triangulations, compare colouring heuristics with exact colouring, implement Vizing's fan argument, and compute chromatic polynomials three ways.

*Before each solution, predict what the output will look like.*

In [ ]:
import itertools, random, math, time
from collections import defaultdict, deque, Counter
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
import sympy as sp
from scipy.spatial import Delaunay
random.seed(7); rng = np.random.default_rng(7)

## Exercises

**P7.1 (★) Euler's formula.** Test random graphs for planarity (networkx implements the left-right algorithm). For the planar ones, traverse the faces of the embedding and check $n-m+f=1+c$ (Theorem 7.5 and Exercise 7.11) and the bound $m\le3n-6$ (Corollary 7.6). How does the probability of planarity of $G(n,p)$ depend on $p$?

**P7.2 (★★) Kuratowski's theorem in action.** For non-planar graphs, extract a Kuratowski subgraph, smooth its vertices of degree $2$, and check that what remains is $K_5$ or $K_{3,3}$ (Theorem 7.12). Check Thomassen's contractible edge lemma (Lemma 7.8) on random $3$-connected graphs.

**P7.3 (★★★) Five colours with Kempe chains.** Generate random triangulations (Delaunay), and implement the proof of Theorem 7.14 as an algorithm: remove a vertex of degree at most $5$, colour recursively, and when the five neighbours use five colours, swap a Kempe chain, choosing the pair of colours using the cyclic order of the neighbours. Check every colouring.

**P7.4 (★★) Colouring heuristics.** Compare greedy colouring in random order, largest-first, smallest-last (degeneracy, Proposition 7.13) and DSATUR with the exact chromatic number found by branch and bound, on random graphs. Check Brooks' theorem (Theorem 7.15) and the interval graphs of Exercise 7.7 (register allocation).

**P7.5 (★★) Vizing's algorithm.** Implement the proof of Theorem 7.16 and colour the edges of random graphs with $\Delta+1$ colours. Then show by exhaustive search that the Petersen graph needs $4$.

**P7.6 (★★) The chromatic polynomial.** Compute $P(G,k)$ by deletion–contraction (Proposition 7.17), by Whitney's broken circuits (Theorem 7.18) and by brute force for small $k$. Count acyclic orientations and compare with $(-1)^nP(G,-1)$ (Theorem 7.19). Finally check the drawing of $K_6$ with three crossings of Exercise 7.4.

## Solutions

### P7.1
**How to find it.** networkx returns a *combinatorial embedding*: the cyclic order of the neighbours around each vertex. A face is traced by following half-edges: from $u\to v$, continue with $v\to w$ where $w$ comes next after $u$ in the order around $v$; each half-edge belongs to exactly one face.

In [ ]:
def faces(emb):
    seen = set(); fs = []
    for u, v in emb.edges():
        if (u, v) in seen: continue
        f = emb.traverse_face(u, v, mark_half_edges=seen)
        fs.append(f)
    return fs

stats = Counter(); worst = 0
for trial in range(2000):
    n = random.randint(3, 30); p = random.choice([0.05, 0.1, 0.15, 0.2])
    G = nx.gnp_random_graph(n, p, seed=random.randrange(10 ** 9))
    planar, emb = nx.check_planarity(G)
    if not planar: stats['non-planar'] += 1; continue
    m = G.number_of_edges(); c = nx.number_connected_components(G)
    ce = sum(1 for comp in nx.connected_components(G) if len(comp) > 1)   # components with an edge
    f_count = len(faces(emb)) - ce + 1      # each such component traces its own copy of the outer face
    assert n - m + f_count == 1 + c, (n, m, f_count, c)
    assert m <= 3 * n - 6
    worst = max(worst, m / (3 * n - 6)); stats['planar'] += 1
print(dict(stats), f'- Euler checked; largest m/(3n-6) seen: {worst:.2f}')

networkx embeds each component separately, so each component with an edge traces its own copy of the outer face: we count the outer face once (isolated vertices have no half-edges and trace nothing). Now the threshold: $G(n,p)$ is planar with high probability when $p=c/n$ with $c<1$ (it is a forest plus a few cycles), and non-planar when $c>1$, as the giant component appears (Tome~3).

In [ ]:
n = 200; cs = np.linspace(0.2, 2.5, 15); prob = []
for cc in cs:
    ok = sum(nx.check_planarity(nx.gnp_random_graph(n, cc / n, seed=s))[0] for s in range(40))
    prob.append(ok / 40)
plt.figure(figsize=(6, 3)); plt.plot(cs, prob, 'o-'); plt.axvline(1, color='gray', lw=0.8)
plt.xlabel('c = np'); plt.ylabel('P(planar)'); plt.title('G(200, c/200)'); plt.tight_layout(); plt.show()

### P7.2
**How to find it.** networkx's `check_planarity(G, counterexample=True)` returns a Kuratowski subgraph. Smoothing a vertex of degree $2$ means replacing its two edges by one; repeat, then test isomorphism with $K_5$ and $K_{3,3}$.

In [ ]:
def smooth(H):
    H = nx.Graph(H)
    changed = True
    while changed:
        changed = False
        for v in list(H.nodes()):
            if H.degree(v) == 2:
                a, b = list(H.neighbors(v))
                if not H.has_edge(a, b):
                    H.remove_node(v); H.add_edge(a, b); changed = True
    H.remove_nodes_from([v for v in list(H.nodes()) if H.degree(v) == 0])
    return H
K5, K33 = nx.complete_graph(5), nx.complete_bipartite_graph(3, 3)
kinds = Counter()
for trial in range(300):
    n = random.randint(6, 40); G = nx.gnm_random_graph(n, random.randint(3 * n - 5, 4 * n), seed=random.randrange(10 ** 9))
    planar, K = nx.check_planarity(G, counterexample=True)
    if planar: continue
    S = smooth(K)
    kind = 'K5' if nx.is_isomorphic(S, K5) else 'K33' if nx.is_isomorphic(S, K33) else '???'
    assert kind != '???' and all(G.has_edge(u, v) for u, v in K.edges())
    kinds[kind] += 1
print('Kuratowski subgraphs found in random non-planar graphs:', dict(kinds))
pet = nx.petersen_graph(); print('Petersen graph:', 'K5' if nx.is_isomorphic(smooth(nx.check_planarity(pet, True)[1]), K5) else 'K33', 'subdivision')

In [ ]:
def contractible_edges(G):
    out = []
    for u, v in G.edges():
        H = nx.contracted_nodes(G, u, v, self_loops=False)
        if nx.node_connectivity(H) >= 3: out.append((u, v))
    return out
for trial in range(40):
    while True:
        n = random.randint(5, 14); G = nx.gnp_random_graph(n, 0.5, seed=random.randrange(10 ** 9))
        if nx.node_connectivity(G) >= 3: break
    ce = contractible_edges(G); assert ce
print('every 3-connected graph tested (5 to 14 vertices) has a contractible edge')
W = nx.wheel_graph(8)
print('wheel W_8: contractible edges are', contractible_edges(W), '(only rim edges: contracting a spoke leaves two rim vertices of degree 2)')

### P7.3
**How to find it.** Delaunay triangulations are planar with straight edges, so the cyclic order of neighbours comes from angles. Recursion: remove a vertex $v$ of degree at most $5$; but the cyclic order of $v$'s neighbours must be taken in the *original* drawing, which never changes since we only delete vertices. Then the proof works verbatim: the chain of colours $c(v_1),c(v_3)$ from $v_1$; if it misses $v_3$ swap it, else swap the chain of $c(v_2),c(v_4)$ from $v_2$.

In [ ]:
def delaunay_graph(N, rng):
    pts = rng.random((N, 2)); tri = Delaunay(pts)
    G = nx.Graph()
    for a, b, c in tri.simplices: G.add_edges_from([(a, b), (b, c), (a, c)])
    return G, pts

def kempe_chain(G, col, start, a, b, alive):
    seen = {start}; st = [start]
    while st:
        u = st.pop()
        for w in G[u]:
            if w in alive and w not in seen and col.get(w) in (a, b): seen.add(w); st.append(w)
    return seen

def five_colour(G, pts):
    order = []; alive = set(G.nodes()); deg = {v: G.degree(v) for v in G}
    while alive:                                   # peel vertices of degree <= 5
        v = next(v for v in alive if deg[v] <= 5)
        order.append(v); alive.remove(v)
        for w in G[v]:
            if w in alive: deg[w] -= 1
    col = {}; alive = set(); swaps = 0
    for v in reversed(order):
        nbrs = [w for w in G[v] if w in alive]
        used = {col[w] for w in nbrs}
        free = [c for c in range(5) if c not in used]
        if not free:                                   # five neighbours, five colours
            nbrs.sort(key=lambda w: math.atan2(pts[w][1] - pts[v][1], pts[w][0] - pts[v][0]))
            v1, v2, v3, v4, v5 = nbrs
            K = kempe_chain(G, col, v1, col[v1], col[v3], alive)
            if v3 not in K: a, b = col[v1], col[v3]
            else:
                K = kempe_chain(G, col, v2, col[v2], col[v4], alive)
                assert v4 not in K                      # Jordan's theorem at work
                a, b = col[v2], col[v4]
            for w in K: col[w] = b if col[w] == a else a
            swaps += 1
            free = [c for c in range(5) if c not in {col[w] for w in nbrs}]
        col[v] = free[0]; alive.add(v)
    return col, swaps

total = 0
for trial in range(30):
    G, pts = delaunay_graph(random.randint(50, 2000), rng)
    col, swaps = five_colour(G, pts); total += swaps
    assert all(col[u] != col[v] for u, v in G.edges()) and max(col.values()) < 5
print(f'30 random triangulations five-coloured and checked; {total} Kempe swaps were needed')

The assertion `v4 not in K` never fails: when the $1$–$3$ chain joins $v_1$ to $v_3$, Jordan's theorem separates $v_2$ from $v_4$. Kempe's 1879 mistake was to do two such swaps simultaneously for four colours; they can interfere.

### P7.4
**How to find it.** DSATUR colours next the vertex whose neighbours already use the most distinct colours. The exact algorithm is the same order with backtracking, pruning when the number of colours reaches the best found so far.

In [ ]:
def greedy(G, order):
    col = {}
    for v in order:
        used = {col[w] for w in G[v] if w in col}
        col[v] = next(c for c in itertools.count() if c not in used)
    return col
def smallest_last(G):
    H = G.copy(); order = []
    while H:
        v = min(H, key=H.degree); order.append(v); H.remove_node(v)
    return order[::-1]
def dsatur(G):
    col = {}; sat = {v: set() for v in G}
    while len(col) < len(G):
        v = max((v for v in G if v not in col), key=lambda v: (len(sat[v]), G.degree(v)))
        c = next(c for c in itertools.count() if c not in sat[v]); col[v] = c
        for w in G[v]: sat[w].add(c)
    return col
def chromatic_number(G):
    best = [max(dsatur(G).values()) + 1 if len(G) else 0]
    nodes = list(G)
    def rec(col, k):
        if k >= best[0]: return
        if len(col) == len(nodes): best[0] = k; return
        v = max((v for v in nodes if v not in col), key=lambda v: (len({col[w] for w in G[v] if w in col}), G.degree(v)))
        used = {col[w] for w in G[v] if w in col}
        for c in range(min(k + 1, best[0] - 1)):
            if c not in used:
                col[v] = c; rec(col, max(k, c + 1)); del col[v]
    rec({}, 0)
    return best[0]

res = defaultdict(list)
for trial in range(150):
    n = random.randint(10, 22); G = nx.gnp_random_graph(n, random.choice([0.2, 0.35, 0.5]), seed=random.randrange(10 ** 9))
    chi = chromatic_number(G)
    res['random order'].append(max(greedy(G, random.sample(list(G), n)).values()) + 1 - chi)
    res['largest first'].append(max(greedy(G, sorted(G, key=G.degree, reverse=True)).values()) + 1 - chi)
    res['smallest last'].append(max(greedy(G, smallest_last(G)).values()) + 1 - chi)
    res['DSATUR'].append(max(dsatur(G).values()) + 1 - chi)
    if nx.is_connected(G) and max(dict(G.degree()).values()) >= 3:
        assert chi <= max(dict(G.degree()).values()) or nx.density(G) == 1                # Brooks
for k, v in res.items():
    print(f'{k:14s}: extra colours over chi, average {np.mean(v):.2f}, worst {max(v)}')

In [ ]:
intervals = []
for i in range(60):
    a = random.uniform(0, 100); intervals.append((a, a + random.expovariate(1 / 12)))
I = nx.Graph(); I.add_nodes_from(range(60))
I.add_edges_from((i, j) for i, j in itertools.combinations(range(60), 2) if intervals[i][0] <= intervals[j][1] and intervals[j][0] <= intervals[i][1])
col = greedy(I, sorted(range(60), key=lambda i: intervals[i][0]))
omega = max(sum(1 for a, b in intervals if a <= x <= b) for x, _ in intervals)
print(f'register allocation: greedy by start uses {max(col.values()) + 1} registers, max number of live variables = {omega}')
fig, ax = plt.subplots(figsize=(7, 3)); cmap = plt.get_cmap('tab20')
for i, (a, b) in enumerate(sorted(intervals)):
    j = intervals.index((a, b)); ax.plot([a, b], [col[j], col[j]], lw=5, color=cmap(i % 20), solid_capstyle='butt')
ax.set_xlabel('time'); ax.set_ylabel('register'); ax.set_title('interval colouring = register allocation'); plt.tight_layout(); plt.show()

### P7.5
**How to find it.** Store the colour of each edge and, for each vertex, the colours present. Build the fan, then follow the three cases of the proof literally; the $\alpha\beta$-path is found by walking.

In [ ]:
def vizing(G):
    D = max(dict(G.degree()).values()); C = list(range(D + 1)); col = {}
    key = lambda a, b: (a, b) if a <= b else (b, a)
    def at(v): return {col[key(v, w)]: w for w in G[v] if key(v, w) in col}
    def missing(v): s = at(v); return next(c for c in C if c not in s)
    def path(start, a, b):                       # maximal path from start, colours a, b, a, ...
        P = [start]; cur, c = start, a
        while True:
            nxt = at(cur).get(c)
            if nxt is None or nxt in P[-2:-1]: return P
            P.append(nxt); cur, c = nxt, (b if c == a else a)
    def swap(P, a, b):
        for u, v in zip(P, P[1:]):
            col[key(u, v)] = b if col[key(u, v)] == a else a
    for x, y0 in G.edges():
        alpha = missing(x); fan = [y0]
        while True:                                # build a maximal fan
            m = missing(fan[-1]); y = at(x).get(m)
            if y is None or y in fan: break
            fan.append(y)
        def shift(j):                              # shift the fan at j
            for i in range(j):
                col[key(x, fan[i])] = col[key(x, fan[i + 1])]
            col.pop(key(x, fan[j]), None)
        beta = missing(fan[-1]); k = len(fan) - 1
        if beta not in at(x):
            shift(k); col[key(x, fan[k])] = beta; continue
        i = fan.index(at(x)[beta])
        saved = dict(col); shift(k)
        if alpha not in at(fan[k]):
            col[key(x, fan[k])] = alpha; continue
        P = path(fan[k], alpha, beta)
        if P[-1] != x:
            swap(P, alpha, beta); col[key(x, fan[k])] = alpha; continue
        col.clear(); col.update(saved); shift(i - 1)
        Q = path(fan[k], alpha, beta)
        assert Q[-1] == fan[i - 1]
        swap(Q, alpha, beta); col[key(x, fan[i - 1])] = alpha
    return col, D

for trial in range(300):
    G = nx.gnp_random_graph(random.randint(2, 30), random.choice([0.1, 0.3, 0.6]), seed=random.randrange(10 ** 9))
    if G.number_of_edges() == 0: continue
    col, D = vizing(G)
    assert len(col) == G.number_of_edges() and max(col.values()) <= D
    for v in G: assert len({col[(min(v, w), max(v, w))] for w in G[v]}) == G.degree(v)
print('300 random graphs edge-coloured with Delta + 1 colours by the fan argument')

In [ ]:
def edge_colourable(G, k):
    E = sorted(G.edges(), key=lambda e: -min(G.degree(e[0]), G.degree(e[1]))); col = {}
    def ok(e, c):
        return all(col.get(f) != c for f in col if set(f) & set(e))
    def rec(i):
        if i == len(E): return True
        for c in range(k):
            if ok(E[i], c):
                col[E[i]] = c
                if rec(i + 1): return True
                del col[E[i]]
        return False
    col[E[0]] = 0
    return rec(1)
t0 = time.time()
print('Petersen graph 3-edge-colourable?', edge_colourable(nx.petersen_graph(), 3), f'({time.time() - t0:.1f} s); 4 colours:', edge_colourable(nx.petersen_graph(), 4))
print('K_7 6-edge-colourable?', edge_colourable(nx.complete_graph(7), 6), '; K_8 7-edge-colourable?', edge_colourable(nx.complete_graph(8), 7))

### P7.6
**How to find it.** Deletion–contraction with memoisation on a canonical form is enough for graphs with up to $10$ or so edges more than vertices. For Whitney, number the edges, enumerate subsets, and test for broken circuits by checking, for each edge $e$ outside the subset $A$, whether $e$ is the smallest edge of a cycle in $A\cup\{e\}$, that is, whether $A$ restricted to edges larger than $e$ joins the ends of $e$ (for every $e$, in $A$ or not).

In [ ]:
k = sp.symbols('k')
def chrom_poly(G):
    G = nx.Graph(G)
    if G.number_of_edges() == 0: return k ** G.number_of_nodes()
    u, v = next(iter(G.edges()))
    H = G.copy(); H.remove_edge(u, v)
    return sp.expand(chrom_poly(H) - chrom_poly(nx.contracted_nodes(H, u, v, self_loops=False)))

def whitney(G):
    E = list(G.edges()); n = G.number_of_nodes(); a = Counter()
    for r in range(len(E) + 1):
        for A in itertools.combinations(range(len(E)), r):
            bad = False
            for e in range(len(E)):                    # is e active: the smallest edge of a cycle C with C - e in A?
                big = [E[i] for i in A if i > e]
                H = nx.Graph(); H.add_nodes_from(G); H.add_edges_from(big)
                if nx.has_path(H, *E[e]): bad = True; break
            if not bad: a[r] += 1
    return sp.expand(sum((-1) ** i * a[i] * k ** (n - i) for i in a))

def brute_count(G, q):
    nodes = list(G)
    return sum(1 for c in itertools.product(range(q), repeat=len(nodes)) if all(c[nodes.index(u)] != c[nodes.index(v)] for u, v in G.edges()))

def acyclic_orientations(G):
    E = list(G.edges()); cnt = 0
    for bits in itertools.product([0, 1], repeat=len(E)):
        D = nx.DiGraph(); D.add_nodes_from(G); D.add_edges_from((u, v) if b else (v, u) for (u, v), b in zip(E, bits))
        cnt += nx.is_directed_acyclic_graph(D)
    return cnt

tests = {'C5': nx.cycle_graph(5), 'K4': nx.complete_graph(4), 'wheel W5': nx.wheel_graph(5), 'cube Q3': nx.hypercube_graph(3),
         'Petersen minus 5 edges': nx.Graph(list(nx.petersen_graph().edges())[:10])}
for name, G in tests.items():
    G = nx.convert_node_labels_to_integers(G)
    P = chrom_poly(G); W = whitney(G) if G.number_of_edges() <= 12 else P
    n = G.number_of_nodes()
    ok_brute = all(P.subs(k, q) == brute_count(G, q) for q in range(4)) if n <= 8 else 'skipped'
    print(f'{name:22s} P = {sp.factor(P)}')
    print(f'{"":22s} Whitney agrees: {sp.expand(P - W) == 0}, brute force q<=3: {ok_brute}, '
          f'(-1)^n P(-1) = {(-1) ** n * P.subs(k, -1)}, acyclic orientations = {acyclic_orientations(G)}')

In [ ]:
n_ = sp.symbols('n')
for n in range(3, 9):
    assert sp.expand(chrom_poly(nx.cycle_graph(n)) - ((k - 1) ** n + (-1) ** n * (k - 1))) == 0
print('P(C_n, k) = (k-1)^n + (-1)^n (k-1) checked for n = 3..8 (Exercise 7.10)')

def crossings(P, edges):
    def o(a, b, c): return (b[0] - a[0]) * (c[1] - a[1]) - (b[1] - a[1]) * (c[0] - a[0])
    def cross(p, q, r, s): return o(p, q, r) * o(p, q, s) < 0 and o(r, s, p) * o(r, s, q) < 0
    return [(e, f) for e, f in itertools.combinations(edges, 2) if len(set(e) | set(f)) == 4 and cross(P[e[0]], P[e[1]], P[f[0]], P[f[1]])]
P6 = [(0, 0), (6, 0), (3, 5.2), (3.3, 2.9), (1.8, 1.4), (3.9, 0.9)]
cr = crossings(P6, list(itertools.combinations(range(6), 2)))
print('K6 drawn with', len(cr), 'crossings:', cr)
plt.figure(figsize=(3.5, 3.2))
for a, b in itertools.combinations(range(6), 2):
    plt.plot([P6[a][0], P6[b][0]], [P6[a][1], P6[b][1]], 'C0', lw=1)
plt.plot(*zip(*P6), 'o', color='C1'); plt.axis('equal'); plt.axis('off'); plt.title('K6 with 3 crossings'); plt.show()